In [1]:
from neo4j import GraphDatabase
from utils.database_utils import generate_database_and_retriever
import pickle
import json

/Users/robertplanas/Documents/GitHub/kg-augmented-multimodal-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("graph_lematized_data.pkl", "rb") as f:
    graph_lematized_data = pickle.load(f)

In [3]:
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "123456789")

In [4]:
from importlib import metadata

from nltk import data


def document_file(document_id, docutments_dict):
    document = json.loads(docutments_dict[document_id].decode("utf-8"))
    document_type = document.get("type", "Unknown type")
    metadata = json.loads(document["metadata"]) if "metadata" in document else {}
    source = metadata.get("filename", "Unknown file")

    return document_type, source


def _add_triplet_to_graph(
    driver, head, tail, relation, confidence=1, head_type="Entity", tail_type="Entity"
):
    """
    Adds a triplet (head, relation, tail) to the Neo4j graph with security sanitization.
    """

    # 1. Sanitize Labels and Relationship Types
    # Cypher does not allow parameters for labels/types, so we must clean the strings.
    safe_head_type = "".join(c for c in head_type if c.isalnum() or c == "_")
    safe_tail_type = "".join(c for c in tail_type if c.isalnum() or c == "_")
    safe_relation = "".join(c for c in relation if c.isalnum() or c == "_")

    # 2. Build the Query
    query = f"""
        MERGE (h:{safe_head_type} {{name: $head_name}})
        MERGE (t:{safe_tail_type} {{name: $tail_name}})
        MERGE (h)-[r:{safe_relation}]->(t)
        ON CREATE SET r.confidence = $conf
        ON MATCH SET r.confidence = $conf
        RETURN h.name, type(r), t.name
    """

    try:
        # 3. Execute the query
        records, summary, keys = driver.execute_query(
            query,
            head_name=head,
            tail_name=tail,
            conf=confidence,
            database_="neo4j",
        )

        # 4. Validation logic
        if summary.counters.contains_updates:
            print(f"Success: ({head})-[{relation}]->({tail}) updated.")
        else:
            print(
                f"No changes: Triplet ({head})-({tail}) already exists with this confidence."
            )

        return records

    except Exception as e:
        print(f"Error adding triplet: {e}")
        return None


def add_relationship(relationship):

    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        head = relationship.head
        tail = relationship.tail
        relation = relationship.relation
        confidence = getattr(relationship, "confidence", 1)
        head_type = getattr(relationship, "head_type", "Entity")
        tail_type = getattr(relationship, "tail_type", "Entity")

        _ = _add_triplet_to_graph(
            driver, head, tail, relation, confidence, head_type, tail_type
        )


def add_file_document_relationship(document_id, document_dic):
    document_type, source = document_file(document_id, document_dic)
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        _ = _add_triplet_to_graph(
            driver,
            head=document_id,
            tail=source,
            relation="BELONGS_TO",
            confidence=1,
            head_type=document_type,
            tail_type="file",
        )


def add_document_to_entity_relationship(document_id, relationship, document_dic):

    document_type, _ = document_file(document_id, document_dic)
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        _ = _add_triplet_to_graph(
            driver,
            head=relationship.head,
            tail=document_id,
            relation="BELONGS_TO",
            confidence=1,
            head_type=relationship.head_type,
            tail_type=document_type,
        )
        _ = _add_triplet_to_graph(
            driver,
            head=relationship.tail,
            tail=document_id,
            relation="BELONGS_TO",
            confidence=1,
            head_type=relationship.tail_type,
            tail_type=document_type,
        )


In [5]:
data_base = "./localdb"
retriever = generate_database_and_retriever(main_folder=data_base)

all_keys = list(retriever.docstore.yield_keys())
all_documents = retriever.docstore.mget(all_keys)
docutments_dic = {all_keys[i]: all_documents[i] for i in range(len(all_keys))}

In [7]:
for document_id in docutments_dic.keys():
    add_file_document_relationship(document_id, docutments_dic)
    for relationship in graph_lematized_data[document_id]:
        add_document_to_entity_relationship(document_id, relationship, docutments_dic)
        add_relationship(relationship)

Success: (ad4f23557b79a1869da0347cd4f8532ac1232a6dd9bb6075a2cbcd4efe41c446)-[BELONGS_TO]->(extract_from_master_thesis_RP.pdf) updated.
Success: (analysis)-[BELONGS_TO]->(ad4f23557b79a1869da0347cd4f8532ac1232a6dd9bb6075a2cbcd4efe41c446) updated.
Success: (model classification error)-[BELONGS_TO]->(ad4f23557b79a1869da0347cd4f8532ac1232a6dd9bb6075a2cbcd4efe41c446) updated.
Success: (analysis)-[PROVIDE_QUALITATIVE_UNDERSTANDING_OF]->(model classification error) updated.
Success: (prediction)-[BELONGS_TO]->(ad4f23557b79a1869da0347cd4f8532ac1232a6dd9bb6075a2cbcd4efe41c446) updated.
Success: (case)-[BELONGS_TO]->(ad4f23557b79a1869da0347cd4f8532ac1232a6dd9bb6075a2cbcd4efe41c446) updated.
Success: (prediction)-[QUANTIFY_HOW]->(case) updated.
Success: (model)-[BELONGS_TO]->(ad4f23557b79a1869da0347cd4f8532ac1232a6dd9bb6075a2cbcd4efe41c446) updated.
Success: (section 4.3.3)-[BELONGS_TO]->(ad4f23557b79a1869da0347cd4f8532ac1232a6dd9bb6075a2cbcd4efe41c446) updated.
Success: (model)-[TRAINED_USING_MET